In [ ]:
# Copyright 2025 by Sysco
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Lab 6.4: Cloud Storage Integration with Vertex AI Workflows that use BigQuery

In this section, we will establish the necessary infrastructure for our ML operations. This includes:
1. Installing the Vertex AI and BigQuery SDKs.
2. Setting up Authentication and Project definitions.
3. Creating a Cloud Storage bucket for artifact management (Model exports, staging).
4. Configuring Service Accounts for pipeline execution.

### Install Vertex AI SDK for Python and other required packages

In [ ]:
# Install the packages
! pip3 install --upgrade --quiet pyarrow \
                                 google-cloud-aiplatform \
                                 google-cloud-bigquery \
                                 google-cloud-bigquery-storage \
                                 db-dtypes

### Authenticate your notebook
Authenticate your environment

In [2]:
import sys

if "google.colab" in sys.modules:

    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information
Learn more about [setting up a project and a development environment.](https://cloud.google.com/vertex-ai/docs/start/cloud-environment)

In [3]:
PROJECT_ID = "mfav2-374520"  # @param {type:"string"}
LOCATION = "us-east1"  # @param {type:"string"}

### Create a Cloud Storage bucket

**Lab 6.4 Key Task:** Create a storage bucket to store intermediate artifacts such as datasets and exported BQML models.

In [ ]:
BUCKET_URI = f"gs://churn-TODO_replacewithyourname-{PROJECT_ID}-unique"  # @param {type:"string"}

**Your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [5]:
! gsutil mb -l {LOCATION} {BUCKET_URI}

Creating gs://churn-user19-mfav2-374520-unique/...


#### Service Account

You use a service account to create Vertex AI Pipeline jobs. If you don't want to use your project's Compute Engine service account, set `SERVICE_ACCOUNT` to another service account ID.

In [6]:
SERVICE_ACCOUNT = "vertex-pipeline-executor@mfav2-374520.iam.gserviceaccount.com"  # @param {type:"string"}

In [8]:
import sys

IS_COLAB = "google.colab" in sys.modules

if (
    SERVICE_ACCOUNT == ""
    or SERVICE_ACCOUNT is None
    or SERVICE_ACCOUNT == "[your-service-account]"
):
    # Get your service account from gcloud
    if not IS_COLAB:
        shell_output = !gcloud auth list 2>/dev/null
        SERVICE_ACCOUNT = shell_output[2].replace("*", "").strip()

    if IS_COLAB:
        shell_output = ! gcloud projects describe $PROJECT_ID
        project_number = shell_output[-1].split(":")[1].strip().replace("'", "")
        SERVICE_ACCOUNT = f"{project_number}-compute@developer.gserviceaccount.com"

# Always print, regardless of condition
print("Service Account:", SERVICE_ACCOUNT)


Service Account: vertex-pipeline-executor@mfav2-374520.iam.gserviceaccount.com


### Import libraries and define constants

In [9]:
import google.cloud.aiplatform as aiplatform
from google.cloud import bigquery

### Initialize Vertex AI and BigQuery SDKs for Python

Initialize the Vertex AI SDK for Python and BigQuery SDK  with your project and the created bucket.

In [10]:
aiplatform.init(project=PROJECT_ID, staging_bucket=BUCKET_URI)

Create the BigQuery client.

In [11]:
bqclient = bigquery.Client(project=PROJECT_ID)

### Set hardware accelerators

You can set hardware accelerators for prediction.

Set the variable `DEPLOY_GPU/DEPLOY_NGPU` to use a container image supporting a GPU and the number of GPUs allocated to the virtual machine (VM) instance. For example, to use a GPU container image with 4 Nvidia Telsa T4 GPUs allocated to each VM, you would specify:

    (aiplatform.AcceleratorType.NVIDIA_TESLA_T4, 4)

Otherwise specify `(None, None)` to use a container image to run on a CPU.

Learn more [about hardware accelerator support for your region.](https://cloud.google.com/vertex-ai/docs/general/locations#accelerators)

In [12]:
import os
from google.cloud import aiplatform

if os.getenv("IS_TESTING_DEPLOY_GPU"):
    # If env var is set, use that GPU count
    DEPLOY_GPU, DEPLOY_NGPU = (
        aiplatform.gapic.AcceleratorType.NVIDIA_TESLA_T4,
        int(os.getenv("IS_TESTING_DEPLOY_GPU")),
    )
else:
    # Default to CPU-only
    DEPLOY_GPU, DEPLOY_NGPU = (None, None)

print("Deployment accelerator:", DEPLOY_GPU, DEPLOY_NGPU)


Deployment accelerator: None None


### Set prebuilt containers

Set the prebuilt Docker container image for prediction.

- Set the variable `TF` to the TensorFlow version of the container image. The following list shows some of the prebuilt images available:


For the latest list, see [prebuilt containers for prediction](https://cloud.google.com/vertex-ai/docs/predictions/pre-built-containers).

In [13]:
import os
from google.cloud import aiplatform

# Example: set TF version from env var or default
if os.getenv("IS_TESTING_TF"):
    TF = os.getenv("IS_TESTING_TF")
else:
    TF = "2.5".replace(".", "-")

# Configure accelerator (None, None for CPU-only)
DEPLOY_GPU, DEPLOY_NGPU = (None, None)

# Build container version string
if TF[0] == "2":
    if DEPLOY_GPU is not None:
        DEPLOY_VERSION = "tf2-gpu.{}".format(TF)
    else:
        DEPLOY_VERSION = "tf2-cpu.{}".format(TF)
else:
    if DEPLOY_GPU is not None:
        DEPLOY_VERSION = "tf-gpu.{}".format(TF)
    else:
        DEPLOY_VERSION = "tf-cpu.{}".format(TF)

# Construct full container URI
DEPLOY_IMAGE = "{}-docker.pkg.dev/vertex-ai/prediction/{}:latest".format(
    LOCATION.split("-")[0], DEPLOY_VERSION
)

print("Deployment:", DEPLOY_IMAGE, DEPLOY_GPU, DEPLOY_NGPU)


Deployment: us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-5:latest None None


### Set machine type

Next, set the machine type to use for prediction.

- Set the variable `DEPLOY_COMPUTE` to configure the compute resources for the VM which is used for prediction.
 - `machine type`
     - `n1-standard`: 3.75GB of memory per vCPU.
     - `n1-highmem`: 6.5GB of memory per vCPU
     - `n1-highcpu`: 0.9 GB of memory per vCPU
 - `vCPUs`: number of \[2, 4, 8, 16, 32, 64, 96 \]

*Note: You may also use n2 and e2 machine types for training and deployment, but they don't support GPUs*

In [14]:
import os

# Pick machine type from env var or default to n1-standard
if os.getenv("IS_TESTING_DEPLOY_MACHINE"):
    MACHINE_TYPE = os.getenv("IS_TESTING_DEPLOY_MACHINE")
else:
    MACHINE_TYPE = "n1-standard"

# Set vCPUs explicitly to 2
VCPU = "2"

# Build full machine type string
DEPLOY_COMPUTE = MACHINE_TYPE + "-" + VCPU

print("Deploy machine type:", DEPLOY_COMPUTE)


Deploy machine type: n1-standard-2


# Lab 6.5: BigQuery ML and Vertex AI Integration Patterns

In this section, we will utilize **BigQuery ML (BQML)** to train models using standard SQL syntax. We will also demonstrate how to integrate these BQML models with Vertex AI for Model Registry and Online Prediction.

## BigQuery ML introduction

BigQuery ML (BQML) provides the capability to train ML tabular models, such as classification and regression in BigQuery using SQL syntax.

Learn more about [BigQuery ML documentation](https://cloud.google.com/bigquery-ml/docs).

In [15]:
IMPORT_FILE = "bq://bigquery-public-data.ml_datasets.penguins"
BQ_TABLE = "bigquery-public-data.ml_datasets.penguins"

### Create BQ dataset resource

First, you create an empty dataset resource in your project.

In [ ]:
BQ_DATASET_NAME = "penguins_todo_replacewithyourname"  # @param {type:"string"}
DATASET_QUERY = f"""CREATE SCHEMA {BQ_DATASET_NAME}
"""

job = bqclient.query(DATASET_QUERY)

### Training and registering the model (Advanced Lab 6.5 & 6.6)

Next, you train the model and automatically register the model to the Vertex AI Model Registry, by adding the following parameters as options:

- `model_registry`: Set to `vertex_ai` to indicate automatic registration to Vertex AI Model Registry.
- `vertex_ai_model_id`: The human readable display name for the registered model.
- `vertex_ai_model_version_aliases`: Alternate name for the model.

In [ ]:
MODEL_NAME = "penguins_todo_replacewithyourname"  # @param {type:"string"}
MODEL_QUERY = f"""
CREATE OR REPLACE MODEL `{BQ_DATASET_NAME}.{MODEL_NAME}`
OPTIONS(
    model_type='DNN_CLASSIFIER',
    labels = ['species'],
    model_registry="vertex_ai",
    vertex_ai_model_id="bqml_model_todo_replacewithyourname",
    vertex_ai_model_version_aliases=["1"]
    )
AS
SELECT *
FROM `{BQ_TABLE}`
"""

job = bqclient.query(MODEL_QUERY)
print(job.errors, job.state)

while job.running():
    from time import sleep

    sleep(30)
    print("Running ...")
print(job.errors, job.state)

try:
    tblname = job.ddl_target_table
    tblname = "{}.{}".format(tblname.dataset_id, tblname.table_id)
    print("{} created in {}".format(tblname, job.ended - job.started))
except Exception as e:
    print(e)

### Lab 6.6: Google Cloud Data and Analytics Architecture for ML utilizing the Vertex AI Model Registry

Finally, you can use the Vertex AI model `list()` method with a filter query to find the automatically registered model.

In [ ]:
from google.cloud import aiplatform

# Initialize in the correct region
aiplatform.init(project=PROJECT_ID, location="us-central1")

# List all models to confirm names
models = aiplatform.Model.list()
for m in models:
    print("Display name:", m.display_name, "| Resource name:", m.resource_name)

# Filter by the actual display name you saw above
filtered_models = aiplatform.Model.list(filter='display_name="bqml_model_todo_replacewithyourname"')

if filtered_models:
    model = filtered_models[0]
    print("\n=== Filtered model details ===")
    print(model.gca_resource)
else:
    print("\nNo models found with that display name in us-central1")


In [ ]:
models = aiplatform.Model.list()
for model in models:
    if model.gca_resource.display_name.startswith("bqml"):
        print(model.gca_resource.display_name)

In [ ]:
EVAL_QUERY = f"""
SELECT *
FROM
  ML.EVALUATE(MODEL {BQ_DATASET_NAME}.{MODEL_NAME})
ORDER BY  roc_auc desc
LIMIT 1"""

try:
    job = bqclient.query(EVAL_QUERY)
    results = job.result().to_dataframe()
    print(results)
except Exception as e:
    print(e)

### Delete the BigQuery ML model

Next, delete the BigQuery ML instance of the model.

In [ ]:
MODEL_QUERY = f"""
DROP MODEL `{BQ_DATASET_NAME}.{MODEL_NAME}`
"""

job = bqclient.query(MODEL_QUERY)

# Cleaning up

To clean up all Google Cloud resources used in this project, you can [delete the Google Cloud
project](https://cloud.google.com/resource-manager/docs/creating-managing-projects#shutting_down_projects) you used for the tutorial.

Otherwise, you can delete the individual resources you created in this tutorial.

Set `delete_storage` to `True` to delete the Cloud Storage bucket used in this notebook.

In [ ]:
# Delete the endpoint using the Vertex endpoint object
try:
    endpoint.undeploy_all()
    endpoint.delete()
except Exception as e:
    print(e)

# Delete the model using the Vertex model object
try:
    model.delete()
except Exception as e:
    print(e)

# Delete the created BigQuery dataset
! bq rm -r -f $PROJECT_ID:$BQ_DATASET_NAME

delete_storage = False
if delete_storage:
    # Delete the created GCS bucket
    ! gsutil rm -r $BUCKET_URI